in this notebook, we will acquire the data needed for the experiment.

we will be uisng the datasets used in the 'enlight' study: `https://github.com/PangeaResearch/enlight-data/tree/main`, specifically the trastuzumab data.


# setup

In [5]:
import os
from pathlib import Path
import pandas as pd, numpy as np
os.environ['BIO_REPPO_ROOT'] = r"C:\Users\Jesse\Google Drive\Learning\freelance\biodata_portfolio"
repo_root = Path(os.environ['BIO_REPPO_ROOT'])

# target data

in the repo, we see that all 5 datasets are breast cancer, and mostly microarray.

| File      | mRNA Value type | Cancer type     | GSE |
|   :---:     |   :----:    |    :---:      |     :---: |         
| Trastuzumab_2.csv | Microarray | Breast | GSE66399 (GSE66305)
| Trastuzumab_3.csv | Microarray  | Breast | GSE37946
|Trastuzumab_4.csv | Microarray  | Breast | GSE42822
|Trastuzumab_5.csv | read counts  | Breast |  10.1038/s41586-021-04278-5
|Trastuzumab.csv | Microarray  | Breast | GSE50948
we also download the 'drug_response_calssification'.
the supplementarty data file in the associated article also lists the source of each experiment. all but the rna-seq are from NCBI GEO. 

when we open GSE66399 in GEO, we see it is an expeirment composed of two parts, where the sub experiment GSE66305 is the rna data we want. 
`https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE66399`

the repo specifies that the files are **raw** data.
so instead of going throug the whole process of normalising and aggregating the experiments, we'll take advantage of a project that did the work for us.


a note on the original data: it's the only project I found where almost all microarray data from geo was uniformly processed and aggreagated. its an exceedingly inefficient file, with data of 400K rna (mostly microarray) samples. 
in seperate work I processed the file into a single zarr database, with a layout similar to the anndata format used in scRNA experiments. 
here, I provide the csv file of the sample metadata, to select the correct rows in the genexp. 

if your really interested, these are the normalisation steps they applied:

Refine bio microarray processing:
Sample level process :
Remove background
Apply SCAN: The  output is log2p1 transformed, and centered [u-0, std=1]

Combine experiments
Filter genes expressed in < 30% and samples with <50% genes expressed
Impute missing values with SVD
Apply quantile normalization - using preprocessor core. 

refine bio 'normalised compendia' for homo sapiens: 
https://www.refine.bio/compendia/normalized

https://docs.refine.bio/en/latest/main_text.html#microarray-pipelines
https://www.bioconductor.org/packages/release/bioc/manuals/SCAN.UPC/man/SCAN.UPC.pdf
https://support.bioconductor.org/p/48544/

# getting row indexes from obs

load the data, 
find indexes associated with the experiment gse's,
flag any duplicates.

In [ ]:
refine_bio_meta = pd.read_csv(repo_root / 'data/rb_obs.csv')
refine_bio_meta

In [7]:
series_list = ["GSE66399", "GSE37946", "GSE42822", "GSE50948"]

mask = refine_bio_meta['experiment_accession'].str.contains("|".join(series_list), na=False)
subset_refine_bio_meta = refine_bio_meta[mask]
subset_refine_bio_meta

,characteristic_BioSourceProvider,characteristic_BioSourceType,experiment_accession,is_microarray,refinebio_accession_code,refinebio_age,refinebio_cell_line,refinebio_compound,refinebio_disease,refinebio_disease_stage,...,refinebio_processor_version,refinebio_race,refinebio_sex,refinebio_source_archive_url,refinebio_source_database,refinebio_specimen_part,refinebio_subject,refinebio_time,refinebio_title,refinebio_treatment
10650,NaN,NaN,GSE42822,True,GSM1050577,32.197,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US001,NaN
10651,NaN,NaN,GSE42822,True,GSM1050578,42.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US005,NaN
10652,NaN,NaN,GSE42822,True,GSM1050579,36.739,NaN,NaN,NaN,NaN,...,v1.23.4-hotfix,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US011,NaN
10653,NaN,NaN,GSE42822,True,GSM1050580,47.099,NaN,NaN,NaN,NaN,...,v1.4.9,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US014,NaN
10654,NaN,NaN,GSE42822,True,GSM1050581,44.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US016,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304139,NaN,NaN,GSE37946,True,GSM930570,72.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,Breast Cancer Tissue 46,NaN
304140,NaN,NaN,GSE37946,True,GSM930571,54.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,Breast Cancer Tissue 47,NaN
304141,NaN,NaN,GSE37946,True,GSM930572,24.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,Breast Cancer Tissue 48,NaN
304142,NaN,NaN,GSE37946,True,GSM930573,62.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,Breast Cancer Tissue 49,NaN


to get a feeling for what fields are actually filled, we'll pull pandas info, and see that it's really only the id columns that we need.

In [14]:
subset_refine_bio_meta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 291 entries, 10650 to 304143
Data columns (total 25 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   characteristic_BioSourceProvider  0 non-null      object 
 1   characteristic_BioSourceType      0 non-null      object 
 2   experiment_accession              291 non-null    object 
 3   is_microarray                     291 non-null    bool   
 4   refinebio_accession_code          291 non-null    object 
 5   refinebio_age                     291 non-null    float64
 6   refinebio_cell_line               0 non-null      object 
 7   refinebio_compound                0 non-null      object 
 8   refinebio_disease                 0 non-null      object 
 9   refinebio_disease_stage           0 non-null      object 
 10  refinebio_organism                291 non-null    object 
 11  refinebio_platform                291 non-null    object 
 12  refine

to be on the safe side we'll use sample id (the `refinebio_accession_code` collumn with the `GSMXXXX` strings) to ensure no duplicates

In [13]:
subset_refine_bio_meta['refinebio_accession_code'].duplicated().any()

False

now we get the indexes as values. 

In [17]:
indexes = np.array(subset_refine_bio_meta.index.values).astype(int)
indexes[:5]

array([10650, 10651, 10652, 10653, 10654])

i used the  indexes to subset the transcriptomic data and saved it to a pickle:
```
genexp = root['layers/refine_qn'].get_orthogonal_selection((indexes, slice(None)))

array([[ 1.72851562e+00, -8.16040039e-02,  7.55468750e+00, ...,
         1.07324219e+00,  1.80786133e-01,  2.46484375e+00],
       ...,
       [ 3.17187500e+00, -9.17053223e-03,  1.47109375e+01, ...,
         2.06835938e+00, -3.30810547e-02,  3.01367188e+00]],
      shape=(291, 14549), dtype=float32)

```

note, I saved the data in zarr in a format optimised for deep learning training where I need to access samples at random, without needing to load the whole dataset into ram. 
therefore, each sample was saved as it own chunk, meaning that i never read more than one sample at a time.


the real dataset is (418073, 14549) and takes 15 GB on disk. and yet, I was able to read the selected data instantly, thanks to correct file structuring. 
```
%timeit genexp = root['layers/refine_qn'].get_orthogonal_selection((indexes, slice(None)))
33.6 ms ± 302 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
```

finally, saved it to pickle for export:
`genexp.dump(repo_root / 'data/rb_trastuzumab_genexp.pickle')`

## pre-process genexp data: QC

plot metrics:
check for Nan, null, inf 
check for all 0 samples
remove genes which are all 0.
print hist of number of genes expressed above some delta (the 1st percentile over whole ds).

# building the file

we also have a csv of the genes used in refine bio (data/rb_gene_vocab). there are 14549, and ecah collumn in the genexp corespons to a gene in the vocab. 
we will use the hugo symbols, since they are easier to read.
to work with csv format, we'll transpose,  assign the symbols as rows, and sample names as collumn names. 


# get the relavant clin data

find the relevant sample id in the clinical data file, 
and keep only samples were we have both genexp and response.
convert response to boolen (1 inidicates response), and keep it in an array named 'labels'

# initial ploting, and color by response

end of file. anything below should be done in different file

# feature selection

use pca to select 256 genes.
later we can add the pca + correlation and balanced selection. 
or selection by correlation boot strap: in each fold, take a subset of genes, and find those correlated with response.